## Assessment 1

Name: Iliana Peters

Student Number: 35723483


### Dataset introduction

The dataset used in this assignment is the Uber Data Analytics Dashboard for the entire 2024 year. It includes the data of various Uber rides, from users and drivers, including dates, locations, timing data, prices, and ratings. A main business case that can be made from this data relates to influential factors in patient satisfaction, leading to the question 'What are the most influential factors that impact rider satisfaction?' Various metrics could be investigated such as ride distance to booking value, pickup duration, vehicle type, and travel duration to ride distance. Additionally, Customer Cancellation Reasons is a free text column that can be textually analysed for trends and additional insights. 

### Part A: Analytical Query Design and Implementation
*Part A assesses your ability to design, implement, and evaluate a non-trivial analytical query using Apache Spark. The focus is not only on producing correct results, but also on demonstrating an understanding of how Spark processes distributed workloads.*

#### 1a. The Business Query
Design and implement a non-trivial business query that requires the following operations:
- A window function, for example, running total, rank within a partition, moving average, cumulative statistics
- High-velocity activity spikes: identify users whose transactions count in any single hour exceeds three standard deviations above their personal hourly mean
- A time-based analysis using date or timestamp attributes

Using the above operations and relating them to the Uber dataset, the following queries are created and will be investigated in the following sections:
1. Calculate statistics on Ride Distance, Booking Value, Pickup Duration, Travel Duration and Driver Rating. These should be ranked by Driver Ratings within the partition. 
2. High-velocity activity spikes: identifying users who has multiple Booking IDs within a single hour that exceeds three standard deviations above the standard user hourly mean. Correlate these to Drive and Customer Ratings.


#### 1b. Dataset Import and Investigation

In [113]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

#setup a spark session and load dataset
master = "local[*]"
app_name = "Uber Business Query DF"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

df = spark.read.csv("ncr_ride_bookings.csv",header=True)
df.show(5)
df.printSchema()
df.count()

+----------+--------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|      Date|    Time|      Booking ID| Booking Status|     Customer ID| Vehicle Type|    Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+----------+--------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+

150000

In [114]:
#convert numerical columns to float from string
from pyspark.sql.functions import col,concat_ws
from pyspark.sql.types import DateType

# Replace string 'null' with true None/null globally
df = df.replace("null", None)

float_col = ["Avg VTAT","Avg CTAT","Ride Distance","Driver Ratings","Customer Ratings","Booking Value"]
df = df.select([col(c).cast("float") if c in float_col else col(c) for c in df.columns])
df = df.withColumn("Datetime",concat_ws(" ", col("Date"), col("Time")).cast("timestamp"))
df.printSchema()
df.show(5)



root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Booking ID: string (nullable = true)
 |-- Booking Status: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Vehicle Type: string (nullable = true)
 |-- Pickup Location: string (nullable = true)
 |-- Drop Location: string (nullable = true)
 |-- Avg VTAT: float (nullable = true)
 |-- Avg CTAT: float (nullable = true)
 |-- Cancelled Rides by Customer: string (nullable = true)
 |-- Reason for cancelling by Customer: string (nullable = true)
 |-- Cancelled Rides by Driver: string (nullable = true)
 |-- Driver Cancellation Reason: string (nullable = true)
 |-- Incomplete Rides: string (nullable = true)
 |-- Incomplete Rides Reason: string (nullable = true)
 |-- Booking Value: float (nullable = true)
 |-- Ride Distance: float (nullable = true)
 |-- Driver Ratings: float (nullable = true)
 |-- Customer Rating: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Datet

In [115]:
#after confirming the Datetime column matches, the seperate Date and Time columns are dropped
df = df.drop("Date", "Time")
df.show(5)

+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+
|      Booking ID| Booking Status|     Customer ID| Vehicle Type|    Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|
+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+---

#### 2. DataFrame Implementation

Implement the query using the Spark DataFrame API

##### 2a. Dataframe Investigation
Below is the full workings to create the queries in using Dataframes. Intermediate steps and working notes will be shown

In [116]:
#BUSINESS QUERY 1 - cumulative statistics on the Ride Distance, Booking Value, Pickup Duration, Travel Duration and Driver Rating.
# These should be ranked by Driver Ratings within the partition. 

print(df.count())
#filter out null ratings and store in cache as intermediate dataframe used for the statistical assessment of query 1
filter_1 = df.dropna(subset=["Driver Ratings"]).cache()
print(filter_1.count())
filter_1.describe("Driver Ratings").show()


150000
93000
+-------+------------------+
|summary|    Driver Ratings|
+-------+------------------+
|  count|             93000|
|   mean| 4.230992466042118|
| stddev|0.4368714755910542|
|    min|               3.0|
|    max|               5.0|
+-------+------------------+



In [117]:
#add a column that would specify the customer rating range into 0-3, 3-4, 4-5
from pyspark.sql.functions import when
filter_1 = filter_1.withColumn("Driver_Rating_Range",when(col("Driver Ratings")<=3.0, "0-3").
                               when(col("Driver Ratings")<=3.5, "3-3.5").when(col("Driver Ratings")<=4.0, "3.5-4").
                               when(col("Driver Ratings")<=4.5, "4-4.5").when(col("Driver Ratings")<=5.0, "4.5-5"))
filter_1.show(20)

+----------------+--------------+----------------+-------------+-------------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+-------------------+
|      Booking ID|Booking Status|     Customer ID| Vehicle Type|    Pickup Location|   Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|Driver_Rating_Range|
+----------------+--------------+----------------+-------------+-------------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+-------------

In [118]:
from pyspark.sql.functions import avg, count, round

stats = (filter_1.groupBy("Driver_Rating_Range").agg(count("*").alias("Count"),round(avg("Driver Ratings"),2).alias("Avg Driver Rating"),
        round(avg("Ride Distance"),2).alias("Avg Ride Distance"),round(avg("Booking Value"),2).alias("Avg Booking Value"),
        round(avg("Avg VTAT"),2).alias("Avg Pickup Duration"),round(avg("Avg CTAT"),2).alias("Avg Travel Duration")))
stats.show()

#clear memory of intermediate dataframe
filter_1.unpersist()

+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|Driver_Rating_Range|Count|Avg Driver Rating|Avg Ride Distance|Avg Booking Value|Avg Pickup Duration|Avg Travel Duration|
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|              3-3.5| 6697|             3.28|            26.15|           507.39|               8.55|              30.02|
|              4.5-5|23444|             4.74|            25.94|           507.81|               8.48|              30.05|
|                0-3|  745|              3.0|            25.41|           521.87|               8.66|              30.12|
|              4-4.5|46540|             4.28|            25.99|           507.57|               8.52|              30.02|
|              3.5-4|15574|              3.8|            26.08|           510.23|               8.53|              30.07|
+-------------------+---

DataFrame[Booking ID: string, Booking Status: string, Customer ID: string, Vehicle Type: string, Pickup Location: string, Drop Location: string, Avg VTAT: float, Avg CTAT: float, Cancelled Rides by Customer: string, Reason for cancelling by Customer: string, Cancelled Rides by Driver: string, Driver Cancellation Reason: string, Incomplete Rides: string, Incomplete Rides Reason: string, Booking Value: float, Ride Distance: float, Driver Ratings: float, Customer Rating: string, Payment Method: string, Datetime: timestamp, Driver_Rating_Range: string]

In [119]:
#BUSINESS QUERY 2 - high-velocity activity spikes: identifyig users who has multiple Booking IDs within a single hour that exceeds 
# three standard deviations above the standard user hourly mean. Correlate these to Drive and Customer Ratings

#count the number of unique Customer IDs in dataset
unique_count = df.select("Customer ID").distinct().count()
total_count = df.count()
#number of enteries with repeat users
print(total_count - unique_count)

#review the Customer IDs with the most bookings
from pyspark.sql.functions import desc
repeat_bookings = (df.groupBy("Customer ID").agg(count("Booking ID").alias("Repeat_Bookings")).filter(col("Repeat_Bookings") > 1)
    .orderBy(desc("Repeat_Bookings")))
repeat_bookings.show()

#average and standard deviation number of repeat bookings 
from pyspark.sql.functions import stddev
repbook_stats = repeat_bookings.agg(avg("Repeat_Bookings").alias("Avg_Repeat_Bookings"),stddev("Repeat_Bookings").alias("StdDev_Repeat_Bookings"))
repbook_stats.show()


1212
+----------------+---------------+
|     Customer ID|Repeat_Bookings|
+----------------+---------------+
|"""CID7828101"""|              3|
|"""CID4523979"""|              3|
|"""CID6715450"""|              3|
|"""CID6468528"""|              3|
|"""CID8727691"""|              3|
|"""CID5481002"""|              3|
|"""CID9329900"""|              2|
|"""CID3448023"""|              2|
|"""CID5312396"""|              2|
|"""CID1798550"""|              2|
|"""CID5911647"""|              2|
|"""CID5896172"""|              2|
|"""CID7611095"""|              2|
|"""CID6792844"""|              2|
|"""CID7117070"""|              2|
|"""CID9028135"""|              2|
|"""CID5945148"""|              2|
|"""CID9173522"""|              2|
|"""CID4140572"""|              2|
|"""CID5521293"""|              2|
+----------------+---------------+
only showing top 20 rows
+-------------------+----------------------+
|Avg_Repeat_Bookings|StdDev_Repeat_Bookings|
+-------------------+-------------------

In [120]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

#dataframe of users with repeat bookings that is cache
repeat_bookings_detail = df.join(repeat_bookings, on="Customer ID", how="inner").cache()
repeat_bookings_detail.count() 

#investigating repeat booking Datetime to see if there are any that happen within the same day and data relating to their bookings
datetime_window = Window.partitionBy("Customer ID").orderBy("Datetime")

#Calculating the time between bookings in hours
velocity_df = (repeat_bookings_detail.withColumn("Prev Booking", lag("Datetime").over(datetime_window)).withColumn("Prev Booking ID",lag("Booking ID").over(datetime_window))
    .withColumn("Time since last Booking",(col("Datetime").cast("long") - col("Prev Booking").cast("long"))/3600))
velocity_df.show()

+----------------+----------------+-------------------+-------------+--------------------+---------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+---------------+-------------------+----------------+-----------------------+
|     Customer ID|      Booking ID|     Booking Status| Vehicle Type|     Pickup Location|  Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|Repeat_Bookings|       Prev Booking| Prev Booking ID|Time since last Booking|
+----------------+----------------+-------------------+-------------+--------------------+------

In [121]:
#investigate the process of join in Spark
repeat_bookings_detail.explain(extended=True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [Customer ID])
:- Project [Booking ID#37741, Booking Status#37742, Customer ID#37743, Vehicle Type#37744, Pickup Location#37745, Drop Location#37746, Avg VTAT#37760, Avg CTAT#37761, Cancelled Rides by Customer#37749, Reason for cancelling by Customer#37750, Cancelled Rides by Driver#37751, Driver Cancellation Reason#37752, Incomplete Rides#37753, Incomplete Rides Reason#37754, Booking Value#37762, Ride Distance#37763, Driver Ratings#37764, Customer Rating#37758, Payment Method#37759, Datetime#37765]
:  +- Project [Date#37739, Time#37740, Booking ID#37741, Booking Status#37742, Customer ID#37743, Vehicle Type#37744, Pickup Location#37745, Drop Location#37746, Avg VTAT#37760, Avg CTAT#37761, Cancelled Rides by Customer#37749, Reason for cancelling by Customer#37750, Cancelled Rides by Driver#37751, Driver Cancellation Reason#37752, Incomplete Rides#37753, Incomplete Rides Reason#37754, Booking Value#37762, Ride Distance#37763, Driver Ratin

In [122]:
#Only showing bookings that are made within 24 hours
velocity_df.filter(col("Time since last Booking") < 24).show()

+----------------+----------------+--------------------+------------+------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+---------------+-------------------+----------------+-----------------------+
|     Customer ID|      Booking ID|      Booking Status|Vehicle Type|   Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|Repeat_Bookings|       Prev Booking| Prev Booking ID|Time since last Booking|
+----------------+----------------+--------------------+------------+------------------+--------

In [123]:
velocity_df.filter(col("Time since last Booking") < 24).select("Customer ID", "Reason for cancelling by Customer", "Driver Cancellation Reason",
                                                               "Time since last Booking").show(truncate=False)

+----------------+--------------------------------------------+-----------------------------------+-----------------------+
|Customer ID     |Reason for cancelling by Customer           |Driver Cancellation Reason         |Time since last Booking|
+----------------+--------------------------------------------+-----------------------------------+-----------------------+
|"""CID1714179"""|NULL                                        |More than permitted people in there|10.0525                |
|"""CID3211810"""|NULL                                        |NULL                               |9.503333333333334      |
|"""CID3422015"""|NULL                                        |Customer related issue             |20.590555555555557     |
|"""CID4021971"""|NULL                                        |NULL                               |1.7222222222222223     |
|"""CID4061291"""|NULL                                        |NULL                               |11.040277777777778     |
|"""CID7

In [124]:
#clear memory of intermediate dataframe
repeat_bookings_detail.unpersist()

DataFrame[Customer ID: string, Booking ID: string, Booking Status: string, Vehicle Type: string, Pickup Location: string, Drop Location: string, Avg VTAT: float, Avg CTAT: float, Cancelled Rides by Customer: string, Reason for cancelling by Customer: string, Cancelled Rides by Driver: string, Driver Cancellation Reason: string, Incomplete Rides: string, Incomplete Rides Reason: string, Booking Value: float, Ride Distance: float, Driver Ratings: float, Customer Rating: string, Payment Method: string, Datetime: timestamp, Repeat_Bookings: bigint]

##### 2b. Dataframe Query Finalisation

Below is the final code for the two business queries in Spark Dataframe API.It is condensed to remove any of the workings and investigative processes done in 2a. This code will be used for the comparative analysis in Part B.

In [141]:
%%time

from pyspark.sql.functions import lag, col, when, count, avg, count, round, desc
from pyspark.sql.window import Window

#Query 1
query_1 = df.dropna(subset=["Driver Ratings"]).cache()
query_1.count()
query_1 = query_1.withColumn("Driver_Rating_Range",when(col("Driver Ratings")<=3.0, "0-3").
                               when(col("Driver Ratings")<=3.5, "3-3.5").when(col("Driver Ratings")<=4.0, "3.5-4").
                               when(col("Driver Ratings")<=4.5, "4-4.5").when(col("Driver Ratings")<=5.0, "4.5-5"))
stats = (query_1.groupBy("Driver_Rating_Range").agg(count("*").alias("Count"),round(avg("Driver Ratings"),2).alias("Avg Driver Rating"),
        round(avg("Ride Distance"),2).alias("Avg Ride Distance"),round(avg("Booking Value"),2).alias("Avg Booking Value"),
        round(avg("Avg VTAT"),2).alias("Avg Pickup Duration"),round(avg("Avg CTAT"),2).alias("Avg Travel Duration")))
print("The Uber distance, timing and price statistics seperated per rating given by Customer")
stats.show()
query_1.unpersist()

#Query 2
query_2_repeat = (df.groupBy("Customer ID").agg(count("Booking ID").alias("Repeat_Bookings")).filter(col("Repeat_Bookings") > 1)
    .orderBy(desc("Repeat_Bookings")))
from pyspark.sql.functions import stddev
repbook_stats = query_2_repeat.agg(avg("Repeat_Bookings").alias("Avg_Repeat_Bookings"),stddev("Repeat_Bookings").
                                    alias("StdDev_Repeat_Bookings"))
print("The average and standard devaition of repeat bookings made in 2024")
repbook_stats.show()
repeat_bookings_detail = df.join(query_2_repeat, on="Customer ID", how="inner").cache()
repeat_bookings_detail.count() 
datetime_window = Window.partitionBy("Customer ID").orderBy("Datetime")
velocity_df = (repeat_bookings_detail.withColumn("Prev Booking", lag("Datetime").over(datetime_window)).withColumn("Prev Booking ID",lag("Booking ID").over(datetime_window))
    .withColumn("Time since last Booking",(col("Datetime").cast("long") - col("Prev Booking").cast("long"))/3600))
print("The Customer ID with multiple bookings made within 24 hours, with booking status, reasoning and time between bookings")
velocity_df.filter(col("Time since last Booking") < 24).select("Customer ID", "Booking Status", "Reason for cancelling by Customer", 
                            "Driver Cancellation Reason","Time since last Booking").show(truncate=False)
repeat_bookings_detail.unpersist()


The Uber distance, timing and price statistics seperated per rating given by Customer
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|Driver_Rating_Range|Count|Avg Driver Rating|Avg Ride Distance|Avg Booking Value|Avg Pickup Duration|Avg Travel Duration|
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|              3-3.5| 6697|             3.28|            26.15|           507.39|               8.55|              30.02|
|              4.5-5|23444|             4.74|            25.94|           507.81|               8.48|              30.05|
|                0-3|  745|              3.0|            25.41|           521.87|               8.66|              30.12|
|              4-4.5|46540|             4.28|            25.99|           507.57|               8.52|              30.02|
|              3.5-4|15574|              3.8|            26.

DataFrame[Customer ID: string, Booking ID: string, Booking Status: string, Vehicle Type: string, Pickup Location: string, Drop Location: string, Avg VTAT: float, Avg CTAT: float, Cancelled Rides by Customer: string, Reason for cancelling by Customer: string, Cancelled Rides by Driver: string, Driver Cancellation Reason: string, Incomplete Rides: string, Incomplete Rides Reason: string, Booking Value: float, Ride Distance: float, Driver Ratings: float, Customer Rating: string, Payment Method: string, Datetime: timestamp, Repeat_Bookings: bigint]

#### 3. Spark SQL Implementation

Implement the same query, or a functionally equivalent version, using Spark SQL

In [136]:
%%time
df.createOrReplaceTempView("df_sql")

#Using similar code with SQL to complete query 1
query_1_sql = spark.sql("""SELECT CASE 
        WHEN `Driver Ratings` <= 3.0 THEN '0-3'
        WHEN `Driver Ratings` <= 3.5 THEN '3-3.5'
        WHEN `Driver Ratings` <= 4.0 THEN '3.5-4'
        WHEN `Driver Ratings` <= 4.5 THEN '4-4.5'
        WHEN `Driver Ratings` <= 5.0 THEN '4.5-5'
        END AS Driver_Rating_Range, 
    COUNT(*) AS Count, ROUND(AVG(`Driver Ratings`), 2) AS `Avg Driver Rating`,
    ROUND(AVG(`Ride Distance`), 2) AS `Avg Ride Distance`, ROUND(AVG(`Booking Value`), 2) AS `Avg Booking Value`,
    ROUND(AVG(`Avg VTAT`), 2) AS `Avg Pickup Duration`,ROUND(AVG(`Avg CTAT`), 2) AS `Avg Travel Duration`
    FROM df_sql WHERE `Driver Ratings` IS NOT NULL GROUP BY Driver_Rating_Range """)

print("The Uber distance, timing and price statistics separated per rating given by Customer")
query_1_sql.show()

#Using similar code with SQL to complete query 2

#create temp view for the subset of data that has repeat bookings
spark.sql("""SELECT `Customer ID`, COUNT(`Booking ID`) AS Repeat_Bookings FROM df_sql
    GROUP BY `Customer ID` HAVING COUNT(`Booking ID`) > 1 ORDER BY Repeat_Bookings DESC""").createOrReplaceTempView("query_2_repeat")

#get stats of average and standard deviation for customers with repreat bookings
repbook_stats_sql = spark.sql(""" SELECT AVG(Repeat_Bookings) AS Avg_Repeat_Bookings, STDDEV(Repeat_Bookings) AS StdDev_Repeat_Bookings
    FROM query_2_repeat """)
print("The average and standard deviation of repeat bookings made in 2024")
repbook_stats_sql.show()

#using an CTE to create dataset based on repeat customers and the time between bookings
velocity_sql = spark.sql("""WITH joined AS (SELECT s.*, r.Repeat_Bookings FROM df_sql s
    INNER JOIN query_2_repeat r ON s.`Customer ID` = r.`Customer ID`), 
    windowed AS (SELECT *, LAG(Datetime) OVER (PARTITION BY `Customer ID` ORDER BY Datetime) AS 
    `Prev Booking`, LAG(`Booking ID`) OVER (PARTITION BY `Customer ID` ORDER BY Datetime) AS 
    `Prev Booking ID` FROM joined) SELECT *, (CAST(Datetime AS LONG) - CAST(`Prev Booking` AS LONG)) 
    / 3600 AS `Time since last Booking` FROM windowed""")

velocity_sql.createOrReplaceTempView("velocity")

print("The Customer ID with multiple bookings made within 24 hours, with booking status, reasoning and time between bookings")
spark.sql("""SELECT `Customer ID`, `Booking Status`, `Reason for cancelling by Customer`, `Driver Cancellation Reason`, 
    `Time since last Booking` FROM velocity WHERE `Time since last Booking` < 24""").show(truncate=False)

The Uber distance, timing and price statistics separated per rating given by Customer
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|Driver_Rating_Range|Count|Avg Driver Rating|Avg Ride Distance|Avg Booking Value|Avg Pickup Duration|Avg Travel Duration|
+-------------------+-----+-----------------+-----------------+-----------------+-------------------+-------------------+
|              3-3.5| 6697|             3.28|            26.15|           507.39|               8.55|              30.02|
|              4.5-5|23444|             4.74|            25.94|           507.81|               8.48|              30.05|
|                0-3|  745|              3.0|            25.41|           521.87|               8.66|              30.12|
|              4-4.5|46540|             4.28|            25.99|           507.57|               8.52|              30.02|
|              3.5-4|15574|              3.8|            26.

#### 4. Result Validation

*Demonstrate that both implementations produce equivalent analytical results. Where minor differences occour due to sorting, formatiing, or floating-point precision, provide a brief explaination.*

The two business queries were chosen to provide information relating to customer satisfaction and if there were any common themes relating to ratings given by customers and repeat bookings. They were sufficiently complicated due to the additional data operations to group per rating range and timing between bookings per customer. Additionally, mathematical operations were required to calculate averages and time-based data for the queries. Various operations and functions were used with both Dataframe API and SQL, however Dataframe API was used to form the structure of the queries and was translated into SQL. 

The output of the queries is impacted by the ordering of the code, as this impacts how much data is utilised during the operation. For example, with operations that only require a single data column, like withColumn(), and therefore only use that data during the operation, which makes it very efficient. Additionally, Spark projects necessary columns back through the code and therefore performs the operation on only those columns. In the second query, the final output is only with 5 columns with the information on multiple bookings.  This would be projected back through the previous operations, like join and window function, to optimise the operating power as much as possible. Finally, the join operation was performed using BroadcastHashJoin, determined through the explain() function. Since the join was performed on data that was already filtered, only using two columns for this join, this type was used as it is ideal for smaller datasets that can fit in the memory; larger datasets a Sort-Merge join may have been used. 

Even though Dataframe API was used initially, the SQL code is inherently easier to read due to the descriptive nature of the queries. Additionally, the queries can be combined into a single code rather than multiple lines with the Dataframe API. This also increases the computing power of SQL; the time to complete the on my CPU Dataframe API query was 1.9s compared to 1.7s with SQL query. Comparing the output of the two versions, the finalised data is identical including floating point precision when calculating the average and standard deviation of repeat booking numbers. Additionally, the sorting between the two output tables is also identical. 


### Part B: System perspective and performance analysis

*Part B shifts the focus from "does the code work" to "why does it work this way." You will need to demonstrate an understanding of Spark's internal execution model by gathering empical data and producing a written analysis*

#### 1. Partitioning Strategy

Investigate the partitioning strategy with hash and range partitioning using the high cardinality column - Booking ID.

##### 1a. Hash Partitioning

In [127]:
from pyspark.sql.functions import spark_partition_id
from pyspark.sql.functions import min, max, avg, stddev

for partitions in range(1, 7):
    df_hashed = df.repartition(partitions, "Booking ID")
    hash_partition_counts = (df_hashed.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
    hash_skew_metrics = hash_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))
    imbalance_ratio = (hash_partition_counts.select((max("row_count") / min("row_count")).alias("imbalance_ratio"))
        .collect()[0]["imbalance_ratio"])

    print(f"Partitions: {partitions} | "f"Hash Imbalance ratio: {imbalance_ratio:.3f}")


Partitions: 1 | Hash Imbalance ratio: 1.000
Partitions: 2 | Hash Imbalance ratio: 1.007
Partitions: 3 | Hash Imbalance ratio: 1.005
Partitions: 4 | Hash Imbalance ratio: 1.015
Partitions: 5 | Hash Imbalance ratio: 1.010
Partitions: 6 | Hash Imbalance ratio: 1.015


In [128]:
#three partitions used as this is still a suitable number to seperate the data and improve computing time while still keeping 
# the dat enteries per partition senesible and meaningful. The dataset is 150,000 entries, so three paritions will not dramatically 
# increase overhead while still able to process the data
partitions = 3

df_hashed = df.repartition(partitions, "Booking ID")
hash_partition_counts = (df_hashed.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
hash_skew_metrics = hash_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))

hash_partition_counts.show()
hash_skew_metrics.show()


+----------+---------+
|Booking ID|row_count|
+----------+---------+
|         0|    49854|
|         1|    50122|
|         2|    50024|
+----------+---------+

+------------------+------------------+------------------+---------------------+
|min_partition_rows|max_partition_rows|avg_partition_rows|stddev_partition_rows|
+------------------+------------------+------------------+---------------------+
|             49854|             50122|           50000.0|   135.60235986147143|
+------------------+------------------+------------------+---------------------+



##### 1b. Range Partitioning

In [129]:
#same number of partitions as ideal hash partition
df_range = df.repartitionByRange(partitions, "Booking ID")

df_range.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").agg(count("*").alias("Row Count")).show()

+----------+---------+
|Booking ID|Row Count|
+----------+---------+
|         0|    48763|
|         1|    50968|
|         2|    50269|
+----------+---------+



In [130]:

range_partition_counts = (df_range.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
range_skew_metrics = range_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))
imbalance_ratio = range_partition_counts.select((max("row_count") / min("row_count")).alias("imbalance_ratio"))


range_partition_counts.show()
range_skew_metrics.show()
imbalance_ratio.show()

+----------+---------+
|Booking ID|row_count|
+----------+---------+
|         0|    48021|
|         1|    55122|
|         2|    46857|
+----------+---------+

+------------------+------------------+------------------+---------------------+
|min_partition_rows|max_partition_rows|avg_partition_rows|stddev_partition_rows|
+------------------+------------------+------------------+---------------------+
|             46902|             54165|           50000.0|   3747.2201696724464|
+------------------+------------------+------------------+---------------------+

+------------------+
|   imbalance_ratio|
+------------------+
|1.1250618213095367|
+------------------+



In [131]:
#Investigate if there is different ideal number of partitions for Range Partitioning. Completed twice as Range Partitioning changes
#each iteration, compared to Hash which is reproducible
for partitions in range(1, 7):
    df_range = df.repartitionByRange(partitions, "Booking ID")
    range_partition_counts = (df_range.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
    range_skew_metrics = range_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))
    imbalance_ratio = (range_partition_counts.select((max("row_count") / min("row_count")).alias("imbalance_ratio"))
        .collect()[0]["imbalance_ratio"])

    print(f"Partitions: {partitions} | "f"Range Imbalance ratio: {imbalance_ratio:.3f}")

Partitions: 1 | Range Imbalance ratio: 1.000
Partitions: 2 | Range Imbalance ratio: 1.026
Partitions: 3 | Range Imbalance ratio: 1.232
Partitions: 4 | Range Imbalance ratio: 1.184
Partitions: 5 | Range Imbalance ratio: 1.175
Partitions: 6 | Range Imbalance ratio: 1.032


In [132]:
for partitions in range(1, 7):
    df_range = df.repartitionByRange(partitions, "Booking ID")
    range_partition_counts = (df_range.withColumn("Booking ID", spark_partition_id()).groupBy("Booking ID").
                            agg(count("*").alias("row_count")))
    range_skew_metrics = range_partition_counts.select(min("row_count").alias("min_partition_rows"),max("row_count").alias("max_partition_rows"),
        avg("row_count").alias("avg_partition_rows"),stddev("row_count").alias("stddev_partition_rows"))
    imbalance_ratio = (range_partition_counts.select((max("row_count") / min("row_count")).alias("imbalance_ratio"))
        .collect()[0]["imbalance_ratio"])

    print(f"Partitions: {partitions} | "f"Range Imbalance ratio: {imbalance_ratio:.3f}")

Partitions: 1 | Range Imbalance ratio: 1.000
Partitions: 2 | Range Imbalance ratio: 1.043
Partitions: 3 | Range Imbalance ratio: 1.183
Partitions: 4 | Range Imbalance ratio: 1.058
Partitions: 5 | Range Imbalance ratio: 1.098
Partitions: 6 | Range Imbalance ratio: 1.229


##### 1c Comparative Analysis

The imbalance ratio, the ratio of maximum number of rows to minimum, was used to investigate skew and ensure the partitioning was as balanced as possible. For the hash partitioning, it was 3 and this is sensible for the dataset size being not overly large but still would benefit from parallel computing. Hash partitioning did have some skew but with a standard deviation between the partitions of 135.6 and an imbalance ratio of 1.005 (which is close to 1.0), the data was distributed evenly. Since Booking ID is set as random and not in an order, hash partitioning works relatively well with this dataset. 

Range partitioning with the same number of partitions shows higher skew effects, with a standard deviation of 1528.1 and imbalance ratio of 1.012. This could lead to an impact with query performance, as a partition with large number of rows would require longer processing time compared to a smaller partition. Since the queries would require all rows to be reviewed and using Booking ID to partition, having higher skew could lead to poorer performance. Interestingly, since the range partitioning estimates the boundaries between partitions each integration, these skew values change each run. This is compared to hash partitioning, which is based on a deterministic mathematical function and reproducible. Above you can see the example of the for loop to investigate imbalance ratios for range partitioning with the ideal partition changing each time. Since the Booking ID is random and not based on a sequential order, this makes range partitioning slightly more uncertain with more variation. Therefore, with this dataset and using Booking ID for partitioning, hash partitioning may be better suited.

#### 2. Execution time benchmarking

Using %%time, the DataFrame API and SQL was run five times and manually the total CPU times recorded below. Information about the operating system was also recorded using the below code. 

$$
\begin{array}{c|ccccc|c}
\text{API} & \text{Run 1 (ms)} & \text{Run 2 (ms)} & \text{Run 3 (ms)} & \text{Run 4 (ms)} & \text{Run 5 (ms)} & \text{Median CPU Total (ms)} \\
\hline
\text{DataFrame} & 27.6 & 19.7 & 16.5 & 19.6 & 24.0 & 19.7 \\
\text{SparkSQL} & 8.42 & 5.9 & 7.93 & 7.03 & 6.78 & 7.03
\end{array}
$$



In [133]:
import platform
import multiprocessing
import pyspark
import psutil

# Python / PySpark
print(f"Python version:       {platform.python_version()}")
print(f"PySpark version:      {pyspark.__version__}")
print(f"Spark master:         {spark.sparkContext.master}")

# Operating system / machine
print(f"Operating system:     {platform.system()} {platform.release()}")
print(f"Architecture:         {platform.machine()}")
print(f"CPU cores available:  {multiprocessing.cpu_count()}")
print(f"System memory: {psutil.virtual_memory().total / (1024**3):.1f} GB")


Python version:       3.13.5
PySpark version:      4.2.0
Spark master:         local[*]
Operating system:     Darwin 25.5.0
Architecture:         arm64
CPU cores available:  10
System memory: 16.0 GB


Based on the timing analysis, the DataFrame API has more than twice the amount of CPU time to process compared to SQL for the same query. This was mainly due to the code itself, as both API types used the same Spark optimiser, which translates the code into logical plans that are then processed by the Catalyst optimiser. One main reason for the increase in time would be the use of caching with the Dataframe API. When a dataset is cached and materialised, it requires an additional Spark operation which is not present in the SQL code. Additionally, the GroupBy with query_2_repeat may be seen as an intermediate step that is not present in the SQL code, further separating the execution time between the APIs. Python/RDD lineage and serialisation overhead would not impact timing for the above codes, which do not require row by row applications with python but instead use Spark functions to optimise the execution. 

#### 3. Execution plan interpretation

*Select one join or aggregation operation from Part A implementation. The DataFrame API is recommended.*

Using the explain() operation, the non-truncated output from the repeat_bookings_detail join is shown below; only the physical plan is reviewed in this discussion. Annotations are added to identify the Shuffle Exchange and operator preceding in both the initial and final plan. In this join, the original dataset is joined based on Customer ID with a dataset formed by GroupBy with only the Customer IDs with multiple books. Since this explain() focuses on both join and aggregation operators, the shuffle explanation will be on the initial aggregation only.

In the plan, there is a HashAggregate preceding and following the Shuffle Exchange. The first HashAggregate computes a partial count, meaning while the second computes a final count with the shuffle in the middle. This means that the first counts are per partition, which is defaulted to 200 as per spark.sql.shuffle.partitions setting, then the shuffle occurs that moves all rows that share a hash key (Customer ID) together, then the final count occurs with the final dataset. The partial count preceding the shuffle allows for optimisation, as only the important factors within this join are the Customer ID and number of times that occurs, therefore this is the only data used and not the entire row. This reduction in data required in the operation leads to a reduction in the network I/O. An impact on task scheduling is the shuffle is the midway point, with operations occurring afterwards waiting on the shuffle to occur. Additionally, the default partition is 200 and from the previous work in B2, this dataset be separated into a lot less partitions and still be suitable. Therefore, the default shuffle could lead to increased overhead for smaller datasets. However, in the final plan, there is '+- AQEShuffleRead coalesced', which highlights the small amount of data per partition and combines them, reducing the overall number of partitions automatically and reducing overhead for small datasets. 

To eliminate the need for the shuffle by pre-partitioning using bucketing with specific number of partitions or setting the number of partitions in the spark.sql.shuffle.partitions manually. Bucketing requires a separate table of data and doesn't change the Physical Plan below. Similarly, setting the number of partitions setting doesn't remove shuffle but does make it more efficient.


In [134]:
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [Customer ID])
:- Project [Booking ID#11147, Booking Status#11148, Customer ID#11149, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171]
:  +- Project [Date#11145, Time#11146, Booking ID#11147, Booking Status#11148, Customer ID#11149, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, cast(concat_ws( , Date#11145, Time#11146) as timestamp) AS Datetime#11171]
:     +- Project [Date#11145, Time#11146, Booking ID#11147, Booking Status#11148, Customer ID#11149, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, cast(Avg VTAT#11153 as float) AS Avg VTAT#11166, cast(Avg CTAT#11154 as float) AS Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, cast(Booking Value#11161 as float) AS Booking Value#11168, cast(Ride Distance#11162 as float) AS Ride Distance#11169, cast(Driver Ratings#11163 as float) AS Driver Ratings#11170, Customer Rating#11164, Payment Method#11165]
:        +- Project [CASE WHEN (Date#11013 = null) THEN cast(null as string) ELSE Date#11013 END AS Date#11145, CASE WHEN (Time#11014 = null) THEN cast(null as string) ELSE Time#11014 END AS Time#11146, CASE WHEN (Booking ID#11015 = null) THEN cast(null as string) ELSE Booking ID#11015 END AS Booking ID#11147, CASE WHEN (Booking Status#11016 = null) THEN cast(null as string) ELSE Booking Status#11016 END AS Booking Status#11148, CASE WHEN (Customer ID#11017 = null) THEN cast(null as string) ELSE Customer ID#11017 END AS Customer ID#11149, CASE WHEN (Vehicle Type#11018 = null) THEN cast(null as string) ELSE Vehicle Type#11018 END AS Vehicle Type#11150, CASE WHEN (Pickup Location#11019 = null) THEN cast(null as string) ELSE Pickup Location#11019 END AS Pickup Location#11151, CASE WHEN (Drop Location#11020 = null) THEN cast(null as string) ELSE Drop Location#11020 END AS Drop Location#11152, CASE WHEN (Avg VTAT#11021 = null) THEN cast(null as string) ELSE Avg VTAT#11021 END AS Avg VTAT#11153, CASE WHEN (Avg CTAT#11022 = null) THEN cast(null as string) ELSE Avg CTAT#11022 END AS Avg CTAT#11154, CASE WHEN (Cancelled Rides by Customer#11023 = null) THEN cast(null as string) ELSE Cancelled Rides by Customer#11023 END AS Cancelled Rides by Customer#11155, CASE WHEN (Reason for cancelling by Customer#11024 = null) THEN cast(null as string) ELSE Reason for cancelling by Customer#11024 END AS Reason for cancelling by Customer#11156, CASE WHEN (Cancelled Rides by Driver#11025 = null) THEN cast(null as string) ELSE Cancelled Rides by Driver#11025 END AS Cancelled Rides by Driver#11157, CASE WHEN (Driver Cancellation Reason#11026 = null) THEN cast(null as string) ELSE Driver Cancellation Reason#11026 END AS Driver Cancellation Reason#11158, CASE WHEN (Incomplete Rides#11027 = null) THEN cast(null as string) ELSE Incomplete Rides#11027 END AS Incomplete Rides#11159, CASE WHEN (Incomplete Rides Reason#11028 = null) THEN cast(null as string) ELSE Incomplete Rides Reason#11028 END AS Incomplete Rides Reason#11160, CASE WHEN (Booking Value#11029 = null) THEN cast(null as string) ELSE Booking Value#11029 END AS Booking Value#11161, CASE WHEN (Ride Distance#11030 = null) THEN cast(null as string) ELSE Ride Distance#11030 END AS Ride Distance#11162, CASE WHEN (Driver Ratings#11031 = null) THEN cast(null as string) ELSE Driver Ratings#11031 END AS Driver Ratings#11163, CASE WHEN (Customer Rating#11032 = null) THEN cast(null as string) ELSE Customer Rating#11032 END AS Customer Rating#11164, CASE WHEN (Payment Method#11033 = null) THEN cast(null as string) ELSE Payment Method#11033 END AS Payment Method#11165]
:           +- Relation [Date#11013,Time#11014,Booking ID#11015,Booking Status#11016,Customer ID#11017,Vehicle Type#11018,Pickup Location#11019,Drop Location#11020,Avg VTAT#11021,Avg CTAT#11022,Cancelled Rides by Customer#11023,Reason for cancelling by Customer#11024,Cancelled Rides by Driver#11025,Driver Cancellation Reason#11026,Incomplete Rides#11027,Incomplete Rides Reason#11028,Booking Value#11029,Ride Distance#11030,Driver Ratings#11031,Customer Rating#11032,Payment Method#11033] csv
+- Sort [Repeat_Bookings#13629L DESC NULLS LAST], true
   +- Filter (Repeat_Bookings#13629L > cast(1 as bigint))
      +- Aggregate [Customer ID#13832], [Customer ID#13832, count(Booking ID#13830) AS Repeat_Bookings#13629L]
         +- Project [Booking ID#13830, Booking Status#13831, Customer ID#13832, Vehicle Type#13833, Pickup Location#13834, Drop Location#13835, Avg VTAT#13849, Avg CTAT#13850, Cancelled Rides by Customer#13838, Reason for cancelling by Customer#13839, Cancelled Rides by Driver#13840, Driver Cancellation Reason#13841, Incomplete Rides#13842, Incomplete Rides Reason#13843, Booking Value#13851, Ride Distance#13852, Driver Ratings#13853, Customer Rating#13847, Payment Method#13848, Datetime#13854]
            +- Project [Date#13828, Time#13829, Booking ID#13830, Booking Status#13831, Customer ID#13832, Vehicle Type#13833, Pickup Location#13834, Drop Location#13835, Avg VTAT#13849, Avg CTAT#13850, Cancelled Rides by Customer#13838, Reason for cancelling by Customer#13839, Cancelled Rides by Driver#13840, Driver Cancellation Reason#13841, Incomplete Rides#13842, Incomplete Rides Reason#13843, Booking Value#13851, Ride Distance#13852, Driver Ratings#13853, Customer Rating#13847, Payment Method#13848, cast(concat_ws( , Date#13828, Time#13829) as timestamp) AS Datetime#13854]
               +- Project [Date#13828, Time#13829, Booking ID#13830, Booking Status#13831, Customer ID#13832, Vehicle Type#13833, Pickup Location#13834, Drop Location#13835, cast(Avg VTAT#13836 as float) AS Avg VTAT#13849, cast(Avg CTAT#13837 as float) AS Avg CTAT#13850, Cancelled Rides by Customer#13838, Reason for cancelling by Customer#13839, Cancelled Rides by Driver#13840, Driver Cancellation Reason#13841, Incomplete Rides#13842, Incomplete Rides Reason#13843, cast(Booking Value#13844 as float) AS Booking Value#13851, cast(Ride Distance#13845 as float) AS Ride Distance#13852, cast(Driver Ratings#13846 as float) AS Driver Ratings#13853, Customer Rating#13847, Payment Method#13848]
                  +- Project [CASE WHEN (Date#13807 = null) THEN cast(null as string) ELSE Date#13807 END AS Date#13828, CASE WHEN (Time#13808 = null) THEN cast(null as string) ELSE Time#13808 END AS Time#13829, CASE WHEN (Booking ID#13809 = null) THEN cast(null as string) ELSE Booking ID#13809 END AS Booking ID#13830, CASE WHEN (Booking Status#13810 = null) THEN cast(null as string) ELSE Booking Status#13810 END AS Booking Status#13831, CASE WHEN (Customer ID#13811 = null) THEN cast(null as string) ELSE Customer ID#13811 END AS Customer ID#13832, CASE WHEN (Vehicle Type#13812 = null) THEN cast(null as string) ELSE Vehicle Type#13812 END AS Vehicle Type#13833, CASE WHEN (Pickup Location#13813 = null) THEN cast(null as string) ELSE Pickup Location#13813 END AS Pickup Location#13834, CASE WHEN (Drop Location#13814 = null) THEN cast(null as string) ELSE Drop Location#13814 END AS Drop Location#13835, CASE WHEN (Avg VTAT#13815 = null) THEN cast(null as string) ELSE Avg VTAT#13815 END AS Avg VTAT#13836, CASE WHEN (Avg CTAT#13816 = null) THEN cast(null as string) ELSE Avg CTAT#13816 END AS Avg CTAT#13837, CASE WHEN (Cancelled Rides by Customer#13817 = null) THEN cast(null as string) ELSE Cancelled Rides by Customer#13817 END AS Cancelled Rides by Customer#13838, CASE WHEN (Reason for cancelling by Customer#13818 = null) THEN cast(null as string) ELSE Reason for cancelling by Customer#13818 END AS Reason for cancelling by Customer#13839, CASE WHEN (Cancelled Rides by Driver#13819 = null) THEN cast(null as string) ELSE Cancelled Rides by Driver#13819 END AS Cancelled Rides by Driver#13840, CASE WHEN (Driver Cancellation Reason#13820 = null) THEN cast(null as string) ELSE Driver Cancellation Reason#13820 END AS Driver Cancellation Reason#13841, CASE WHEN (Incomplete Rides#13821 = null) THEN cast(null as string) ELSE Incomplete Rides#13821 END AS Incomplete Rides#13842, CASE WHEN (Incomplete Rides Reason#13822 = null) THEN cast(null as string) ELSE Incomplete Rides Reason#13822 END AS Incomplete Rides Reason#13843, CASE WHEN (Booking Value#13823 = null) THEN cast(null as string) ELSE Booking Value#13823 END AS Booking Value#13844, CASE WHEN (Ride Distance#13824 = null) THEN cast(null as string) ELSE Ride Distance#13824 END AS Ride Distance#13845, CASE WHEN (Driver Ratings#13825 = null) THEN cast(null as string) ELSE Driver Ratings#13825 END AS Driver Ratings#13846, CASE WHEN (Customer Rating#13826 = null) THEN cast(null as string) ELSE Customer Rating#13826 END AS Customer Rating#13847, CASE WHEN (Payment Method#13827 = null) THEN cast(null as string) ELSE Payment Method#13827 END AS Payment Method#13848]
                     +- Relation [Date#13807,Time#13808,Booking ID#13809,Booking Status#13810,Customer ID#13811,Vehicle Type#13812,Pickup Location#13813,Drop Location#13814,Avg VTAT#13815,Avg CTAT#13816,Cancelled Rides by Customer#13817,Reason for cancelling by Customer#13818,Cancelled Rides by Driver#13819,Driver Cancellation Reason#13820,Incomplete Rides#13821,Incomplete Rides Reason#13822,Booking Value#13823,Ride Distance#13824,Driver Ratings#13825,Customer Rating#13826,Payment Method#13827] csv

== Analyzed Logical Plan ==
Customer ID: string, Booking ID: string, Booking Status: string, Vehicle Type: string, Pickup Location: string, Drop Location: string, Avg VTAT: float, Avg CTAT: float, Cancelled Rides by Customer: string, Reason for cancelling by Customer: string, Cancelled Rides by Driver: string, Driver Cancellation Reason: string, Incomplete Rides: string, Incomplete Rides Reason: string, Booking Value: float, Ride Distance: float, Driver Ratings: float, Customer Rating: string, Payment Method: string, Datetime: timestamp, Repeat_Bookings: bigint
Project [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L]
+- Join Inner, (Customer ID#11149 = Customer ID#13832)
   :- Project [Booking ID#11147, Booking Status#11148, Customer ID#11149, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171]
   :  +- Project [Date#11145, Time#11146, Booking ID#11147, Booking Status#11148, Customer ID#11149, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, cast(concat_ws( , Date#11145, Time#11146) as timestamp) AS Datetime#11171]
   :     +- Project [Date#11145, Time#11146, Booking ID#11147, Booking Status#11148, Customer ID#11149, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, cast(Avg VTAT#11153 as float) AS Avg VTAT#11166, cast(Avg CTAT#11154 as float) AS Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, cast(Booking Value#11161 as float) AS Booking Value#11168, cast(Ride Distance#11162 as float) AS Ride Distance#11169, cast(Driver Ratings#11163 as float) AS Driver Ratings#11170, Customer Rating#11164, Payment Method#11165]
   :        +- Project [CASE WHEN (Date#11013 = null) THEN cast(null as string) ELSE Date#11013 END AS Date#11145, CASE WHEN (Time#11014 = null) THEN cast(null as string) ELSE Time#11014 END AS Time#11146, CASE WHEN (Booking ID#11015 = null) THEN cast(null as string) ELSE Booking ID#11015 END AS Booking ID#11147, CASE WHEN (Booking Status#11016 = null) THEN cast(null as string) ELSE Booking Status#11016 END AS Booking Status#11148, CASE WHEN (Customer ID#11017 = null) THEN cast(null as string) ELSE Customer ID#11017 END AS Customer ID#11149, CASE WHEN (Vehicle Type#11018 = null) THEN cast(null as string) ELSE Vehicle Type#11018 END AS Vehicle Type#11150, CASE WHEN (Pickup Location#11019 = null) THEN cast(null as string) ELSE Pickup Location#11019 END AS Pickup Location#11151, CASE WHEN (Drop Location#11020 = null) THEN cast(null as string) ELSE Drop Location#11020 END AS Drop Location#11152, CASE WHEN (Avg VTAT#11021 = null) THEN cast(null as string) ELSE Avg VTAT#11021 END AS Avg VTAT#11153, CASE WHEN (Avg CTAT#11022 = null) THEN cast(null as string) ELSE Avg CTAT#11022 END AS Avg CTAT#11154, CASE WHEN (Cancelled Rides by Customer#11023 = null) THEN cast(null as string) ELSE Cancelled Rides by Customer#11023 END AS Cancelled Rides by Customer#11155, CASE WHEN (Reason for cancelling by Customer#11024 = null) THEN cast(null as string) ELSE Reason for cancelling by Customer#11024 END AS Reason for cancelling by Customer#11156, CASE WHEN (Cancelled Rides by Driver#11025 = null) THEN cast(null as string) ELSE Cancelled Rides by Driver#11025 END AS Cancelled Rides by Driver#11157, CASE WHEN (Driver Cancellation Reason#11026 = null) THEN cast(null as string) ELSE Driver Cancellation Reason#11026 END AS Driver Cancellation Reason#11158, CASE WHEN (Incomplete Rides#11027 = null) THEN cast(null as string) ELSE Incomplete Rides#11027 END AS Incomplete Rides#11159, CASE WHEN (Incomplete Rides Reason#11028 = null) THEN cast(null as string) ELSE Incomplete Rides Reason#11028 END AS Incomplete Rides Reason#11160, CASE WHEN (Booking Value#11029 = null) THEN cast(null as string) ELSE Booking Value#11029 END AS Booking Value#11161, CASE WHEN (Ride Distance#11030 = null) THEN cast(null as string) ELSE Ride Distance#11030 END AS Ride Distance#11162, CASE WHEN (Driver Ratings#11031 = null) THEN cast(null as string) ELSE Driver Ratings#11031 END AS Driver Ratings#11163, CASE WHEN (Customer Rating#11032 = null) THEN cast(null as string) ELSE Customer Rating#11032 END AS Customer Rating#11164, CASE WHEN (Payment Method#11033 = null) THEN cast(null as string) ELSE Payment Method#11033 END AS Payment Method#11165]
   :           +- Relation [Date#11013,Time#11014,Booking ID#11015,Booking Status#11016,Customer ID#11017,Vehicle Type#11018,Pickup Location#11019,Drop Location#11020,Avg VTAT#11021,Avg CTAT#11022,Cancelled Rides by Customer#11023,Reason for cancelling by Customer#11024,Cancelled Rides by Driver#11025,Driver Cancellation Reason#11026,Incomplete Rides#11027,Incomplete Rides Reason#11028,Booking Value#11029,Ride Distance#11030,Driver Ratings#11031,Customer Rating#11032,Payment Method#11033] csv
   +- Sort [Repeat_Bookings#13629L DESC NULLS LAST], true
      +- Filter (Repeat_Bookings#13629L > cast(1 as bigint))
         +- Aggregate [Customer ID#13832], [Customer ID#13832, count(Booking ID#13830) AS Repeat_Bookings#13629L]
            +- Project [Booking ID#13830, Booking Status#13831, Customer ID#13832, Vehicle Type#13833, Pickup Location#13834, Drop Location#13835, Avg VTAT#13849, Avg CTAT#13850, Cancelled Rides by Customer#13838, Reason for cancelling by Customer#13839, Cancelled Rides by Driver#13840, Driver Cancellation Reason#13841, Incomplete Rides#13842, Incomplete Rides Reason#13843, Booking Value#13851, Ride Distance#13852, Driver Ratings#13853, Customer Rating#13847, Payment Method#13848, Datetime#13854]
               +- Project [Date#13828, Time#13829, Booking ID#13830, Booking Status#13831, Customer ID#13832, Vehicle Type#13833, Pickup Location#13834, Drop Location#13835, Avg VTAT#13849, Avg CTAT#13850, Cancelled Rides by Customer#13838, Reason for cancelling by Customer#13839, Cancelled Rides by Driver#13840, Driver Cancellation Reason#13841, Incomplete Rides#13842, Incomplete Rides Reason#13843, Booking Value#13851, Ride Distance#13852, Driver Ratings#13853, Customer Rating#13847, Payment Method#13848, cast(concat_ws( , Date#13828, Time#13829) as timestamp) AS Datetime#13854]
                  +- Project [Date#13828, Time#13829, Booking ID#13830, Booking Status#13831, Customer ID#13832, Vehicle Type#13833, Pickup Location#13834, Drop Location#13835, cast(Avg VTAT#13836 as float) AS Avg VTAT#13849, cast(Avg CTAT#13837 as float) AS Avg CTAT#13850, Cancelled Rides by Customer#13838, Reason for cancelling by Customer#13839, Cancelled Rides by Driver#13840, Driver Cancellation Reason#13841, Incomplete Rides#13842, Incomplete Rides Reason#13843, cast(Booking Value#13844 as float) AS Booking Value#13851, cast(Ride Distance#13845 as float) AS Ride Distance#13852, cast(Driver Ratings#13846 as float) AS Driver Ratings#13853, Customer Rating#13847, Payment Method#13848]
                     +- Project [CASE WHEN (Date#13807 = null) THEN cast(null as string) ELSE Date#13807 END AS Date#13828, CASE WHEN (Time#13808 = null) THEN cast(null as string) ELSE Time#13808 END AS Time#13829, CASE WHEN (Booking ID#13809 = null) THEN cast(null as string) ELSE Booking ID#13809 END AS Booking ID#13830, CASE WHEN (Booking Status#13810 = null) THEN cast(null as string) ELSE Booking Status#13810 END AS Booking Status#13831, CASE WHEN (Customer ID#13811 = null) THEN cast(null as string) ELSE Customer ID#13811 END AS Customer ID#13832, CASE WHEN (Vehicle Type#13812 = null) THEN cast(null as string) ELSE Vehicle Type#13812 END AS Vehicle Type#13833, CASE WHEN (Pickup Location#13813 = null) THEN cast(null as string) ELSE Pickup Location#13813 END AS Pickup Location#13834, CASE WHEN (Drop Location#13814 = null) THEN cast(null as string) ELSE Drop Location#13814 END AS Drop Location#13835, CASE WHEN (Avg VTAT#13815 = null) THEN cast(null as string) ELSE Avg VTAT#13815 END AS Avg VTAT#13836, CASE WHEN (Avg CTAT#13816 = null) THEN cast(null as string) ELSE Avg CTAT#13816 END AS Avg CTAT#13837, CASE WHEN (Cancelled Rides by Customer#13817 = null) THEN cast(null as string) ELSE Cancelled Rides by Customer#13817 END AS Cancelled Rides by Customer#13838, CASE WHEN (Reason for cancelling by Customer#13818 = null) THEN cast(null as string) ELSE Reason for cancelling by Customer#13818 END AS Reason for cancelling by Customer#13839, CASE WHEN (Cancelled Rides by Driver#13819 = null) THEN cast(null as string) ELSE Cancelled Rides by Driver#13819 END AS Cancelled Rides by Driver#13840, CASE WHEN (Driver Cancellation Reason#13820 = null) THEN cast(null as string) ELSE Driver Cancellation Reason#13820 END AS Driver Cancellation Reason#13841, CASE WHEN (Incomplete Rides#13821 = null) THEN cast(null as string) ELSE Incomplete Rides#13821 END AS Incomplete Rides#13842, CASE WHEN (Incomplete Rides Reason#13822 = null) THEN cast(null as string) ELSE Incomplete Rides Reason#13822 END AS Incomplete Rides Reason#13843, CASE WHEN (Booking Value#13823 = null) THEN cast(null as string) ELSE Booking Value#13823 END AS Booking Value#13844, CASE WHEN (Ride Distance#13824 = null) THEN cast(null as string) ELSE Ride Distance#13824 END AS Ride Distance#13845, CASE WHEN (Driver Ratings#13825 = null) THEN cast(null as string) ELSE Driver Ratings#13825 END AS Driver Ratings#13846, CASE WHEN (Customer Rating#13826 = null) THEN cast(null as string) ELSE Customer Rating#13826 END AS Customer Rating#13847, CASE WHEN (Payment Method#13827 = null) THEN cast(null as string) ELSE Payment Method#13827 END AS Payment Method#13848]
                        +- Relation [Date#13807,Time#13808,Booking ID#13809,Booking Status#13810,Customer ID#13811,Vehicle Type#13812,Pickup Location#13813,Drop Location#13814,Avg VTAT#13815,Avg CTAT#13816,Cancelled Rides by Customer#13817,Reason for cancelling by Customer#13818,Cancelled Rides by Driver#13819,Driver Cancellation Reason#13820,Incomplete Rides#13821,Incomplete Rides Reason#13822,Booking Value#13823,Ride Distance#13824,Driver Ratings#13825,Customer Rating#13826,Payment Method#13827] csv

== Optimized Logical Plan ==
InMemoryRelation [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L], StorageLevel(disk, memory, deserialized, 1 replicas)
   +- AdaptiveSparkPlan isFinalPlan=true
      +- == Final Plan ==
         ResultQueryStage 2
         +- *(3) Project [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L]
            +- *(3) BroadcastHashJoin [Customer ID#11149], [Customer ID#13832], Inner, BuildRight, false, false
               :- *(3) Project [CASE WHEN (Booking ID#11015 = null) THEN null ELSE Booking ID#11015 END AS Booking ID#11147, CASE WHEN (Booking Status#11016 = null) THEN null ELSE Booking Status#11016 END AS Booking Status#11148, CASE WHEN (Customer ID#11017 = null) THEN null ELSE Customer ID#11017 END AS Customer ID#11149, CASE WHEN (Vehicle Type#11018 = null) THEN null ELSE Vehicle Type#11018 END AS Vehicle Type#11150, CASE WHEN (Pickup Location#11019 = null) THEN null ELSE Pickup Location#11019 END AS Pickup Location#11151, CASE WHEN (Drop Location#11020 = null) THEN null ELSE Drop Location#11020 END AS Drop Location#11152, CASE WHEN (Avg VTAT#11021 = null) THEN null ELSE cast(Avg VTAT#11021 as float) END AS Avg VTAT#11166, CASE WHEN (Avg CTAT#11022 = null) THEN null ELSE cast(Avg CTAT#11022 as float) END AS Avg CTAT#11167, CASE WHEN (Cancelled Rides by Customer#11023 = null) THEN null ELSE Cancelled Rides by Customer#11023 END AS Cancelled Rides by Customer#11155, CASE WHEN (Reason for cancelling by Customer#11024 = null) THEN null ELSE Reason for cancelling by Customer#11024 END AS Reason for cancelling by Customer#11156, CASE WHEN (Cancelled Rides by Driver#11025 = null) THEN null ELSE Cancelled Rides by Driver#11025 END AS Cancelled Rides by Driver#11157, CASE WHEN (Driver Cancellation Reason#11026 = null) THEN null ELSE Driver Cancellation Reason#11026 END AS Driver Cancellation Reason#11158, CASE WHEN (Incomplete Rides#11027 = null) THEN null ELSE Incomplete Rides#11027 END AS Incomplete Rides#11159, CASE WHEN (Incomplete Rides Reason#11028 = null) THEN null ELSE Incomplete Rides Reason#11028 END AS Incomplete Rides Reason#11160, CASE WHEN (Booking Value#11029 = null) THEN null ELSE cast(Booking Value#11029 as float) END AS Booking Value#11168, CASE WHEN (Ride Distance#11030 = null) THEN null ELSE cast(Ride Distance#11030 as float) END AS Ride Distance#11169, CASE WHEN (Driver Ratings#11031 = null) THEN null ELSE cast(Driver Ratings#11031 as float) END AS Driver Ratings#11170, CASE WHEN (Customer Rating#11032 = null) THEN null ELSE Customer Rating#11032 END AS Customer Rating#11164, CASE WHEN (Payment Method#11033 = null) THEN null ELSE Payment Method#11033 END AS Payment Method#11165, cast(concat_ws( , CASE WHEN (Date#11013 = null) THEN null ELSE Date#11013 END, CASE WHEN (Time#11014 = null) THEN null ELSE Time#11014 END) as timestamp) AS Datetime#11171]
               :  +- *(3) Filter CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END
               :     +- FileScan csv [Date#11013,Time#11014,Booking ID#11015,Booking Status#11016,Customer ID#11017,Vehicle Type#11018,Pickup Location#11019,Drop Location#11020,Avg VTAT#11021,Avg CTAT#11022,Cancelled Rides by Customer#11023,Reason for cancelling by Customer#11024,Cancelled Rides by Driver#11025,Driver Cancellation Reason#11026,Incomplete Rides#11027,Incomplete Rides Reason#11028,Booking Value#11029,Ride Distance#11030,Driver Ratings#11031,Customer Rating#11032,Payment Method#11033] Batched: false, DataFilters: [CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Date:string,Time:string,Booking ID:string,Booking Status:string,Customer ID:string,Vehicle...
               +- BroadcastQueryStage 1
                  +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=6447]
                     +- *(2) Filter (Repeat_Bookings#13629L > 1)
                        +- *(2) HashAggregate(keys=[Customer ID#13832], functions=[count(Booking ID#13830)], output=[Customer ID#13832, Repeat_Bookings#13629L])
                           +- AQEShuffleRead coalesced
                              +- ShuffleQueryStage 0
                                 +- Exchange hashpartitioning(Customer ID#13832, 200), ENSURE_REQUIREMENTS, [plan_id=6399]
                                    +- *(1) HashAggregate(keys=[Customer ID#13832], functions=[partial_count(Booking ID#13830)], output=[Customer ID#13832, count#13857L])
                                       +- *(1) Project [CASE WHEN (Booking ID#13809 = null) THEN null ELSE Booking ID#13809 END AS Booking ID#13830, CASE WHEN (Customer ID#13811 = null) THEN null ELSE Customer ID#13811 END AS Customer ID#13832]
                                          +- *(1) Filter CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END
                                             +- FileScan csv [Booking ID#13809,Customer ID#13811] Batched: false, DataFilters: [CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Booking ID:string,Customer ID:string>
      +- == Initial Plan ==
         Project [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L]
         +- BroadcastHashJoin [Customer ID#11149], [Customer ID#13832], Inner, BuildRight, false, false
            :- Project [CASE WHEN (Booking ID#11015 = null) THEN null ELSE Booking ID#11015 END AS Booking ID#11147, CASE WHEN (Booking Status#11016 = null) THEN null ELSE Booking Status#11016 END AS Booking Status#11148, CASE WHEN (Customer ID#11017 = null) THEN null ELSE Customer ID#11017 END AS Customer ID#11149, CASE WHEN (Vehicle Type#11018 = null) THEN null ELSE Vehicle Type#11018 END AS Vehicle Type#11150, CASE WHEN (Pickup Location#11019 = null) THEN null ELSE Pickup Location#11019 END AS Pickup Location#11151, CASE WHEN (Drop Location#11020 = null) THEN null ELSE Drop Location#11020 END AS Drop Location#11152, CASE WHEN (Avg VTAT#11021 = null) THEN null ELSE cast(Avg VTAT#11021 as float) END AS Avg VTAT#11166, CASE WHEN (Avg CTAT#11022 = null) THEN null ELSE cast(Avg CTAT#11022 as float) END AS Avg CTAT#11167, CASE WHEN (Cancelled Rides by Customer#11023 = null) THEN null ELSE Cancelled Rides by Customer#11023 END AS Cancelled Rides by Customer#11155, CASE WHEN (Reason for cancelling by Customer#11024 = null) THEN null ELSE Reason for cancelling by Customer#11024 END AS Reason for cancelling by Customer#11156, CASE WHEN (Cancelled Rides by Driver#11025 = null) THEN null ELSE Cancelled Rides by Driver#11025 END AS Cancelled Rides by Driver#11157, CASE WHEN (Driver Cancellation Reason#11026 = null) THEN null ELSE Driver Cancellation Reason#11026 END AS Driver Cancellation Reason#11158, CASE WHEN (Incomplete Rides#11027 = null) THEN null ELSE Incomplete Rides#11027 END AS Incomplete Rides#11159, CASE WHEN (Incomplete Rides Reason#11028 = null) THEN null ELSE Incomplete Rides Reason#11028 END AS Incomplete Rides Reason#11160, CASE WHEN (Booking Value#11029 = null) THEN null ELSE cast(Booking Value#11029 as float) END AS Booking Value#11168, CASE WHEN (Ride Distance#11030 = null) THEN null ELSE cast(Ride Distance#11030 as float) END AS Ride Distance#11169, CASE WHEN (Driver Ratings#11031 = null) THEN null ELSE cast(Driver Ratings#11031 as float) END AS Driver Ratings#11170, CASE WHEN (Customer Rating#11032 = null) THEN null ELSE Customer Rating#11032 END AS Customer Rating#11164, CASE WHEN (Payment Method#11033 = null) THEN null ELSE Payment Method#11033 END AS Payment Method#11165, cast(concat_ws( , CASE WHEN (Date#11013 = null) THEN null ELSE Date#11013 END, CASE WHEN (Time#11014 = null) THEN null ELSE Time#11014 END) as timestamp) AS Datetime#11171]
            :  +- Filter CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END
            :     +- FileScan csv [Date#11013,Time#11014,Booking ID#11015,Booking Status#11016,Customer ID#11017,Vehicle Type#11018,Pickup Location#11019,Drop Location#11020,Avg VTAT#11021,Avg CTAT#11022,Cancelled Rides by Customer#11023,Reason for cancelling by Customer#11024,Cancelled Rides by Driver#11025,Driver Cancellation Reason#11026,Incomplete Rides#11027,Incomplete Rides Reason#11028,Booking Value#11029,Ride Distance#11030,Driver Ratings#11031,Customer Rating#11032,Payment Method#11033] Batched: false, DataFilters: [CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Date:string,Time:string,Booking ID:string,Booking Status:string,Customer ID:string,Vehicle...
            +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=6356]
               +- Filter (Repeat_Bookings#13629L > 1)
                  +- HashAggregate(keys=[Customer ID#13832], functions=[count(Booking ID#13830)], output=[Customer ID#13832, Repeat_Bookings#13629L])
                     +- Exchange hashpartitioning(Customer ID#13832, 200), ENSURE_REQUIREMENTS, [plan_id=6352]
                        +- HashAggregate(keys=[Customer ID#13832], functions=[partial_count(Booking ID#13830)], output=[Customer ID#13832, count#13857L])
                           +- Project [CASE WHEN (Booking ID#13809 = null) THEN null ELSE Booking ID#13809 END AS Booking ID#13830, CASE WHEN (Customer ID#13811 = null) THEN null ELSE Customer ID#13811 END AS Customer ID#13832]
                              +- Filter CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END
                                 +- FileScan csv [Booking ID#13809,Customer ID#13811] Batched: false, DataFilters: [CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Booking ID:string,Customer ID:string>

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L]
      +- InMemoryRelation [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 2
                  +- *(3) Project [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L]
                     +- *(3) BroadcastHashJoin [Customer ID#11149], [Customer ID#13832], Inner, BuildRight, false, false
                        :- *(3) Project [CASE WHEN (Booking ID#11015 = null) THEN null ELSE Booking ID#11015 END AS Booking ID#11147, CASE WHEN (Booking Status#11016 = null) THEN null ELSE Booking Status#11016 END AS Booking Status#11148, CASE WHEN (Customer ID#11017 = null) THEN null ELSE Customer ID#11017 END AS Customer ID#11149, CASE WHEN (Vehicle Type#11018 = null) THEN null ELSE Vehicle Type#11018 END AS Vehicle Type#11150, CASE WHEN (Pickup Location#11019 = null) THEN null ELSE Pickup Location#11019 END AS Pickup Location#11151, CASE WHEN (Drop Location#11020 = null) THEN null ELSE Drop Location#11020 END AS Drop Location#11152, CASE WHEN (Avg VTAT#11021 = null) THEN null ELSE cast(Avg VTAT#11021 as float) END AS Avg VTAT#11166, CASE WHEN (Avg CTAT#11022 = null) THEN null ELSE cast(Avg CTAT#11022 as float) END AS Avg CTAT#11167, CASE WHEN (Cancelled Rides by Customer#11023 = null) THEN null ELSE Cancelled Rides by Customer#11023 END AS Cancelled Rides by Customer#11155, CASE WHEN (Reason for cancelling by Customer#11024 = null) THEN null ELSE Reason for cancelling by Customer#11024 END AS Reason for cancelling by Customer#11156, CASE WHEN (Cancelled Rides by Driver#11025 = null) THEN null ELSE Cancelled Rides by Driver#11025 END AS Cancelled Rides by Driver#11157, CASE WHEN (Driver Cancellation Reason#11026 = null) THEN null ELSE Driver Cancellation Reason#11026 END AS Driver Cancellation Reason#11158, CASE WHEN (Incomplete Rides#11027 = null) THEN null ELSE Incomplete Rides#11027 END AS Incomplete Rides#11159, CASE WHEN (Incomplete Rides Reason#11028 = null) THEN null ELSE Incomplete Rides Reason#11028 END AS Incomplete Rides Reason#11160, CASE WHEN (Booking Value#11029 = null) THEN null ELSE cast(Booking Value#11029 as float) END AS Booking Value#11168, CASE WHEN (Ride Distance#11030 = null) THEN null ELSE cast(Ride Distance#11030 as float) END AS Ride Distance#11169, CASE WHEN (Driver Ratings#11031 = null) THEN null ELSE cast(Driver Ratings#11031 as float) END AS Driver Ratings#11170, CASE WHEN (Customer Rating#11032 = null) THEN null ELSE Customer Rating#11032 END AS Customer Rating#11164, CASE WHEN (Payment Method#11033 = null) THEN null ELSE Payment Method#11033 END AS Payment Method#11165, cast(concat_ws( , CASE WHEN (Date#11013 = null) THEN null ELSE Date#11013 END, CASE WHEN (Time#11014 = null) THEN null ELSE Time#11014 END) as timestamp) AS Datetime#11171]
                        :  +- *(3) Filter CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END
                        :     +- FileScan csv [Date#11013,Time#11014,Booking ID#11015,Booking Status#11016,Customer ID#11017,Vehicle Type#11018,Pickup Location#11019,Drop Location#11020,Avg VTAT#11021,Avg CTAT#11022,Cancelled Rides by Customer#11023,Reason for cancelling by Customer#11024,Cancelled Rides by Driver#11025,Driver Cancellation Reason#11026,Incomplete Rides#11027,Incomplete Rides Reason#11028,Booking Value#11029,Ride Distance#11030,Driver Ratings#11031,Customer Rating#11032,Payment Method#11033] Batched: false, DataFilters: [CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Date:string,Time:string,Booking ID:string,Booking Status:string,Customer ID:string,Vehicle...
                        +- BroadcastQueryStage 1
                           +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=6447]
                              +- *(2) Filter (Repeat_Bookings#13629L > 1)
   #Part B, 3. requirement. In the Physical Plan, this is the operator following the Shuffle - HashAggregate (2)
                                 +- *(2) HashAggregate(keys=[Customer ID#13832], functions=[count(Booking ID#13830)], output=[Customer ID#13832, Repeat_Bookings#13629L])
                                    +- AQEShuffleRead coalesced
                                       +- ShuffleQueryStage 0
    #Part B, 3. requirement. In the Physical Plan, below is the Exchange hash partitioning
                                          +- Exchange hashpartitioning(Customer ID#13832, 200), ENSURE_REQUIREMENTS, [plan_id=6399]
   #Part B, 3. requirement. In the Physical Plan, this is the operator preceeding the Shuffle - HashAggregate (1)      
                                             +- *(1) HashAggregate(keys=[Customer ID#13832], functions=[partial_count(Booking ID#13830)], output=[Customer ID#13832, count#13857L])
                                                +- *(1) Project [CASE WHEN (Booking ID#13809 = null) THEN null ELSE Booking ID#13809 END AS Booking ID#13830, CASE WHEN (Customer ID#13811 = null) THEN null ELSE Customer ID#13811 END AS Customer ID#13832]
                                                   +- *(1) Filter CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END
                                                      +- FileScan csv [Booking ID#13809,Customer ID#13811] Batched: false, DataFilters: [CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Booking ID:string,Customer ID:string>
               +- == Initial Plan ==
                  Project [Customer ID#11149, Booking ID#11147, Booking Status#11148, Vehicle Type#11150, Pickup Location#11151, Drop Location#11152, Avg VTAT#11166, Avg CTAT#11167, Cancelled Rides by Customer#11155, Reason for cancelling by Customer#11156, Cancelled Rides by Driver#11157, Driver Cancellation Reason#11158, Incomplete Rides#11159, Incomplete Rides Reason#11160, Booking Value#11168, Ride Distance#11169, Driver Ratings#11170, Customer Rating#11164, Payment Method#11165, Datetime#11171, Repeat_Bookings#13629L]
                  +- BroadcastHashJoin [Customer ID#11149], [Customer ID#13832], Inner, BuildRight, false, false
                     :- Project [CASE WHEN (Booking ID#11015 = null) THEN null ELSE Booking ID#11015 END AS Booking ID#11147, CASE WHEN (Booking Status#11016 = null) THEN null ELSE Booking Status#11016 END AS Booking Status#11148, CASE WHEN (Customer ID#11017 = null) THEN null ELSE Customer ID#11017 END AS Customer ID#11149, CASE WHEN (Vehicle Type#11018 = null) THEN null ELSE Vehicle Type#11018 END AS Vehicle Type#11150, CASE WHEN (Pickup Location#11019 = null) THEN null ELSE Pickup Location#11019 END AS Pickup Location#11151, CASE WHEN (Drop Location#11020 = null) THEN null ELSE Drop Location#11020 END AS Drop Location#11152, CASE WHEN (Avg VTAT#11021 = null) THEN null ELSE cast(Avg VTAT#11021 as float) END AS Avg VTAT#11166, CASE WHEN (Avg CTAT#11022 = null) THEN null ELSE cast(Avg CTAT#11022 as float) END AS Avg CTAT#11167, CASE WHEN (Cancelled Rides by Customer#11023 = null) THEN null ELSE Cancelled Rides by Customer#11023 END AS Cancelled Rides by Customer#11155, CASE WHEN (Reason for cancelling by Customer#11024 = null) THEN null ELSE Reason for cancelling by Customer#11024 END AS Reason for cancelling by Customer#11156, CASE WHEN (Cancelled Rides by Driver#11025 = null) THEN null ELSE Cancelled Rides by Driver#11025 END AS Cancelled Rides by Driver#11157, CASE WHEN (Driver Cancellation Reason#11026 = null) THEN null ELSE Driver Cancellation Reason#11026 END AS Driver Cancellation Reason#11158, CASE WHEN (Incomplete Rides#11027 = null) THEN null ELSE Incomplete Rides#11027 END AS Incomplete Rides#11159, CASE WHEN (Incomplete Rides Reason#11028 = null) THEN null ELSE Incomplete Rides Reason#11028 END AS Incomplete Rides Reason#11160, CASE WHEN (Booking Value#11029 = null) THEN null ELSE cast(Booking Value#11029 as float) END AS Booking Value#11168, CASE WHEN (Ride Distance#11030 = null) THEN null ELSE cast(Ride Distance#11030 as float) END AS Ride Distance#11169, CASE WHEN (Driver Ratings#11031 = null) THEN null ELSE cast(Driver Ratings#11031 as float) END AS Driver Ratings#11170, CASE WHEN (Customer Rating#11032 = null) THEN null ELSE Customer Rating#11032 END AS Customer Rating#11164, CASE WHEN (Payment Method#11033 = null) THEN null ELSE Payment Method#11033 END AS Payment Method#11165, cast(concat_ws( , CASE WHEN (Date#11013 = null) THEN null ELSE Date#11013 END, CASE WHEN (Time#11014 = null) THEN null ELSE Time#11014 END) as timestamp) AS Datetime#11171]
                     :  +- Filter CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END
                     :     +- FileScan csv [Date#11013,Time#11014,Booking ID#11015,Booking Status#11016,Customer ID#11017,Vehicle Type#11018,Pickup Location#11019,Drop Location#11020,Avg VTAT#11021,Avg CTAT#11022,Cancelled Rides by Customer#11023,Reason for cancelling by Customer#11024,Cancelled Rides by Driver#11025,Driver Cancellation Reason#11026,Incomplete Rides#11027,Incomplete Rides Reason#11028,Booking Value#11029,Ride Distance#11030,Driver Ratings#11031,Customer Rating#11032,Payment Method#11033] Batched: false, DataFilters: [CASE WHEN (Customer ID#11017 = null) THEN false ELSE isnotnull(Customer ID#11017) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Date:string,Time:string,Booking ID:string,Booking Status:string,Customer ID:string,Vehicle...
                     +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=6356]
                        +- Filter (Repeat_Bookings#13629L > 1)
   #Part B, 3. requirement. In the initial plan, the HashAgrregate is the second immediately following the shuffle exchange
                           +- HashAggregate(keys=[Customer ID#13832], functions=[count(Booking ID#13830)], output=[Customer ID#13832, Repeat_Bookings#13629L])
   #Part B, 3. requirement. The shuffle exchange is below                          
                             +- Exchange hashpartitioning(Customer ID#13832, 200), ENSURE_REQUIREMENTS, [plan_id=6352]
   #Part B, 3. requirement. In the initial plan, the HashAgrregate is the first immediately preceeding the shuffle exchange                  
                                 +- HashAggregate(keys=[Customer ID#13832], functions=[partial_count(Booking ID#13830)], output=[Customer ID#13832, count#13857L])
                                    +- Project [CASE WHEN (Booking ID#13809 = null) THEN null ELSE Booking ID#13809 END AS Booking ID#13830, CASE WHEN (Customer ID#13811 = null) THEN null ELSE Customer ID#13811 END AS Customer ID#13832]
                                       +- Filter CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END
                                          +- FileScan csv [Booking ID#13809,Customer ID#13811] Batched: false, DataFilters: [CASE WHEN (Customer ID#13811 = null) THEN false ELSE isnotnull(Customer ID#13811) END], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ilianapeters/Documents/Documents - Iliana’s MacBook Pro/Mo..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Booking ID:string,Customer ID:string>



SyntaxError: unterminated string literal (detected at line 2) (2634448106.py, line 2)

#### 4. Spark Web UI

*You need to analyse the DAG (Directed Acyclic Graph) visualation generated by your DataFrame or SparkSQL implementation using Spark Web UI. A DAG visually represents the sequence of Spark operations, their dependencies, and the stage boundries created during execution.*



In [135]:
print(spark.sparkContext.uiWebUrl)

http://192.168.1.118:4040


The following is the DAG from the code discussed in Part B Section 3: *repeat_bookings_detail = df.join(repeat_bookings, on="Customer ID", how="inner").cache* and *repeat_bookings_detail.count()*

![DAG Screenshot](./Repeat%20Booking%20Detail%20DAG%20Job.png)

![DAG Screenshot SQL Query](./Repeat%20Booking%20Detail%20DAG%20SQL%20Query.png)

![DAG Screenshot Stage](./Repeat%20Booking%20Detail%20Stages.png)

From the above screenshots, there are two stages involved with this query. Stage 905 is skipped, as it is being called from outputs completed previously in the code but the stage is separated by Exchange, shuffling the data between stages. However, if we look at the associated SQL query 191, there are five stages involved in this query with two main branches. Before the branches are joined, a lot of the operations are similar, Scan csv, filter and project, which means this would be optimised to reduce this computing cost. For example, if the original dataset was cached it would not have to be read in again each time it's called. This improvement can be seen from the timing of two initial blue boxes, with the left branch taking 1.6s and the right branch taking 1.0s. This computing time could be greatly reduced if this operation did not need to occour multiple times throughout the code and could be called whenever needed. 

In the first branch, after the data is shuffled once removing null entries and AQEShuffleRead reads the data back in. This is the final count, as discussed in Section 3, combining the data after partitioning when completing the GroupBy aggregation with the creation of repeat_bookings. At BroadcastExchange, the seperation occours to start the joining of the data with 1206 repeat Customer IDs found and at BroadcastHashJoin the data is joined with the associated 2418 rows that have the Booking IDs for these Customer IDs. Since repeat_booking_detail is cache to optimise memory storage and efficiency, additional stages are completed to add to memory with InMemoryTableScan and count() activating the cache. Regarding skew, as discussed previously, due to the smaller size of the dataset Spark assesses the degree of partitioning required and combines small partitions as required. Interesting with this dataset of 150,000 entries, all partitions are combined into a single partition. This is seen between Stage 900 and Stage 902, with skew seen in Stage 900 but in the next stage all data is within a single partition. 

![Stage 900 Data](./Stage%20900%20Metric.png)

![Stage 902 Data](./Stage%20902%20Metric.png)

Reviewing the ShuffleReadSize for Stage 900, there is skew with a minimum partition having only 2179 records while the median partition size has 24609 records. However, the record in the partitions is defined as too small and automatically combined into a single partition in the next stage, eliminating skew for this specific dataset. For a larger dataset, there would be more of an impact on computing time and overhead. 